# Food Memory - Retrieval-First Image Classification

This notebook is generated from `course/build_notebook.py` so the learning
path stays reproducible.

The thesis is the same as Digit Memory: before training a model, build a
memory and measure it. The larger Food-101 setting changes the representation
from raw pixels to CLIP embeddings and the index from a small KDTree to FAISS.

## 1. Architecture

```
query image
    |
    v
canonical RGB hash -> exact hit -> stored label
    |
   miss
    |
    v
CLIP image embedding -> FAISS vector search -> k-NN vote
```

Exact memory answers repeats. Vector search answers new but visually similar
food images. Both paths report latency and neighbor evidence.

In [ ]:
from pathlib import Path
import json

import numpy as np
from PIL import Image

from food_memory.hashing import image_digest, array_digest
from food_memory.metrics import l2_normalize

print("Food Memory imports loaded.")

## 2. Exact Image Hashing

The exact path uses decoded RGB pixels and dimensions, not raw file bytes.
That makes the hash stable for the in-memory image object while still strict
enough that recompression, crop, or color changes fall through to retrieval.

In [ ]:
image = Image.new("RGB", (8, 8), color=(220, 80, 40))
same = Image.new("RGB", (8, 8), color=(220, 80, 40))
changed = Image.new("RGB", (8, 8), color=(220, 80, 41))

print(image_digest(image) == image_digest(same))
print(image_digest(image) == image_digest(changed))

## 3. Embedding Space

In the full project, images are embedded with
`openai/clip-vit-base-patch32`. Embeddings are normalized so inner product is
cosine similarity. This lets FAISS use fast maximum-inner-product search.

In [ ]:
toy = np.array([[3.0, 4.0], [1.0, 0.0]], dtype=np.float32)
print(l2_normalize(toy))
print(array_digest(l2_normalize(toy)))

## 4. Build And Evaluate

Use quick mode first:

```bash
python -m food_memory.build --mode quick --batch-size 16
python -m food_memory.evaluate --mode quick --index hnsw --k 1 3 5
python -m food_memory.bench --artifact-dir artifacts/quick
```

Then run full mode when you are ready to spend the download and embedding
time:

```bash
python -m food_memory.build --mode full --batch-size 32
python -m food_memory.evaluate --mode full --index hnsw --k 1 3 5
```

## 5. Reading The Results

Look for four things:

1. Retrieval top-1/top-5 accuracy.
2. HNSW speed versus flat FAISS search.
3. CLIP zero-shot and logistic-regression baselines.
4. Neighbor examples that make successes and failures inspectable.

If retrieval is competitive, you have an explainable baseline with no deep
fine-tuning. If it fails, the failure cases tell you what representation or
model training needs to fix.